# Early-Exit CNN → Quantised TFLite → C

**Author:** Ajay Kumar M

Flow:
1. Fine-tune a pretrained lightweight CNN on a 10-class classification task (CIFAR-10)
2. Attach early-exit heads at 3 intermediate stage boundaries
3. Train exits jointly with weighted cross-entropy loss
4. Quantise to INT8 TFLite
5. Compile to C source via Apache TVM


## 0 · Install dependencies

In [ ]:
!pip install -q tensorflow==2.12.0 keras==2.12.0 tflite==2.1.0 numpy==1.23.5
!pip install -q apache-tvm==0.14.dev273


Restart session to load the dependencies. Some errors may be seen when installing the TCM version. They can be ignored.

## 1 · Configuration

Edit **`MODEL_KEY`** to switch architectures.


In [ ]:
import os, pathlib, shutil, urllib.request, tarfile, pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tvm, tvm.relay as relay
from tvm.relay.backend import Executor, Runtime
from tvm.micro import export_model_library_format

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Dataset
IMAGE_SIZE      = (64, 64, 3)
BATCH_SIZE      = 32
NUM_CLASSES     = 10
CIFAR_CLASSES   = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
UART_SAMPLES    = 50                    # samples per class for UART test export
CACHE_DIR       = "/content/drive/MyDrive/cached_datasets"

# Training
LEARNING_RATE   = 5e-4
EPOCHS_FINETUNE = 10
EPOCHS_EE       = 10
DROPOUT_RATE    = 0.1

# Early-exit
EE_LOSS_WEIGHTS = [0.2, 0.4, 0.7, 1.0]   # weight per exit (last = final head)

# Exit placement
AUTO_SELECT_EXITS = True                 # False = use exit_layers from registry
                                         # True  = run algorithm to find optimal exits

# TVM
TVM_OPT_LEVEL = 3
QUANTIZE_FOR_TVM = True                   # Set to True for quantized, False for unquantized C code

# Fine-tune control
FINETUNE_FROM_SCRATCH = True             # True  = train backbone from imagenet
                                         # False = load saved weights from Drive
FINETUNED_WEIGHTS_DIR = "/content/drive/MyDrive/finetuned_weights"


## 2 · Load data

In [ ]:
MODEL_REGISTRY = {

    "mobilenet_v1": {
        "name": "MobileNetV1",
        "build_fn": lambda: tf.keras.applications.MobileNet(
            input_shape=IMAGE_SIZE, alpha=0.25, weights="imagenet", include_top=False
        ),
        "exit_layers": ["conv_pw_2_relu", "conv_pw_4_relu", "conv_pw_6_relu"],
        "head_offset": -5,
    },

    "mobilenet_v2": {
        "name": "MobileNetV2",
        "build_fn": lambda: tf.keras.applications.MobileNetV2(
            input_shape=IMAGE_SIZE, alpha=0.35, weights="imagenet", include_top=False
        ),
        "exit_layers": ["block_3_expand_relu", "block_6_expand_relu",
                        "block_10_expand_relu"],
        "head_offset": -1,
    },

    "mobilenet_v3_small": {
        "name": "MobileNetV3Small",
        "build_fn": lambda: tf.keras.applications.MobileNetV3Small(
            input_shape=IMAGE_SIZE, alpha=0.75, weights="imagenet",
            include_top=False, include_preprocessing=False,
        ),
        "exit_layers": ["expanded_conv_3/project",
                        "expanded_conv_6/project",
                        "expanded_conv_8/project"],
        "head_offset": -1,
    },

    "efficientnet_b0": {
        "name": "EfficientNetB0",
        "build_fn": lambda: tf.keras.applications.EfficientNetB0(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        "exit_layers": ["block2a_expand_activation", "block3a_expand_activation",
                        "block5a_expand_activation"],
        "head_offset": -1,
        "expects_0_255": True,    # flag: this model preprocesses internally
    },

    "resnet50": {
        "name": "ResNet50",
        "build_fn": lambda: tf.keras.applications.ResNet50(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        "exit_layers": ["conv2_block3_out", "conv3_block4_out", "conv4_block6_out"],
        "head_offset": -1,
    },

    "densenet121": {
        "name": "DenseNet121",
        "build_fn": lambda: tf.keras.applications.DenseNet121(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        "exit_layers": ["pool2_conv", "pool3_conv", "pool4_conv"],
        "head_offset": -1,
    },

    "vgg16": {
        "name": "VGG16",
        "build_fn": lambda: tf.keras.applications.VGG16(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        "exit_layers": ["block2_pool", "block3_pool", "block4_pool"],
        "head_offset": -1,
    },

    "nasnet_mobile": {
        "name": "NASNetMobile",
        "build_fn": lambda: tf.keras.applications.NASNetMobile(
            input_shape=IMAGE_SIZE, weights="imagenet", include_top=False,
        ),
        # NASNet cells: stem → 4 Normal → Reduction → 4 Normal → Reduction → 4 Normal
        # Run backbone.summary() to verify layer names for your TF version.
        "exit_layers": ["normal_concat_3", "normal_concat_6",
                        "normal_concat_9"],
        "head_offset": -1,
        "tvm_custom_pipeline": False,  # NASNet's branchy graph makes the
        "tvm_opt_level": 1,            # custom fold-before-QNN pipeline hang.
                                       # Standard pipeline compiles fine.
    },

    "efficientnet_v2_b0": {
        "name": "EfficientNetV2B0",
        "build_fn": lambda: tf.keras.applications.EfficientNetV2B0(
            input_shape=IMAGE_SIZE, weights="imagenet",
            include_top=False, include_preprocessing=False,
        ),
        # EfficientNetV2B0 stages: block1 → block2 → ... → block6
        # Exit points at ~25/50/75% depth across stages.
        "exit_layers": ["block2b_add", "block4a_expand_activation",
                        "block5e_add"],
        "head_offset": -1,
    },

}


In [ ]:
# ━━━━━━━━━  CHOOSE YOUR MODEL HERE  ━━━━━━━━━━━━━━━━━━━━━━━━
MODEL_KEY = "mobilenet_v2"
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

MODEL_CFG  = MODEL_REGISTRY[MODEL_KEY]
OUTPUT_DIR = pathlib.Path(f"./output_{MODEL_CFG['name']}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Model: {MODEL_CFG['name']}   Output: {OUTPUT_DIR}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ── Try loading cached arrays from Drive ─────────────────
cache_path = os.path.join(CACHE_DIR, f"cifar{NUM_CLASSES}_{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}.npz")
os.makedirs(CACHE_DIR, exist_ok=True)

if os.path.exists(cache_path):
    print(f"Loading cached dataset from {cache_path} ...")
    data = np.load(cache_path)
    x_train_all, y_train_all = data["x_train"], data["y_train"]
    x_test_all, y_test_all   = data["x_test"],  data["y_test"]
    print(f"  train: {len(x_train_all)},  test holdout: {len(x_test_all)}")

else:
    print("Cache not found — building dataset from scratch ...")
    (x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = tf.keras.datasets.cifar10.load_data()

    # Rescale to [0,1]
    x_train_all = x_train_raw.astype("float32") / 255.0
    y_train_all = y_train_raw.squeeze()
    x_test_all  = x_test_raw.astype("float32") / 255.0
    y_test_all  = y_test_raw.squeeze()

    # Resize 32x32 → IMAGE_SIZE
    x_train_all = tf.image.resize(x_train_all, IMAGE_SIZE[:2]).numpy()
    x_test_all  = tf.image.resize(x_test_all, IMAGE_SIZE[:2]).numpy()

    # One-hot encode
    y_train_all = tf.keras.utils.to_categorical(y_train_all, NUM_CLASSES)
    y_test_all  = tf.keras.utils.to_categorical(y_test_all, NUM_CLASSES)

    # Save to Drive for next time
    np.savez_compressed(cache_path,
                        x_train=x_train_all, y_train=y_train_all,
                        x_test=x_test_all,   y_test=y_test_all)
    print(f"Cached processed arrays → {cache_path}")

# Shuffle training data and split 80/20 → train/val
idx = np.random.permutation(len(x_train_all))
x_train_all, y_train_all = x_train_all[idx], y_train_all[idx]
split = int(0.8 * len(x_train_all))

train_ds = tf.data.Dataset.from_tensor_slices(
    (x_train_all[:split], y_train_all[:split])
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices(
    (x_train_all[split:], y_train_all[split:])
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
full_ds = tf.data.Dataset.from_tensor_slices(
    (x_train_all, y_train_all)
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Test set
test_images = x_test_all
test_labels = y_test_all

print(f"CIFAR-10 — {NUM_CLASSES} classes: {CIFAR_CLASSES}")
print(f"train: {len(train_ds)} batches,  val: {len(val_ds)} batches,  full: {len(full_ds)} batches")
print(f"Test set: {len(test_images)} images (from original CIFAR-10 test set)")


## 3 · Fine-tune the base backbone

In [ ]:
pretrained = MODEL_CFG["build_fn"]()

# ── Build the backbone model ─────────────────────────────
backbone_model = tf.keras.Model(
    inputs=pretrained.inputs,
    outputs=pretrained.layers[MODEL_CFG["head_offset"]].output,
)

layers = [tf.keras.layers.InputLayer(input_shape=IMAGE_SIZE)]
if MODEL_CFG.get("expects_0_255"):
    layers.append(tf.keras.layers.Rescaling(255.0))

layers += [
    backbone_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(DROPOUT_RATE),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
]
base_model = tf.keras.Sequential(layers)

base_model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss="categorical_crossentropy", metrics=["accuracy"],
)

weights_path = os.path.join(FINETUNED_WEIGHTS_DIR,
                            f"{MODEL_CFG['name']}_cifar{NUM_CLASSES}_finetuned.weights.h5")

if FINETUNE_FROM_SCRATCH:
    print(f"Fine-tuning {MODEL_CFG['name']} from ImageNet weights ...")
    history = base_model.fit(train_ds, validation_data=val_ds,
                             epochs=EPOCHS_FINETUNE, verbose=2)
else:
    print(f"Loading saved weights from {weights_path} ...")
    base_model.load_weights(weights_path)
    history = None
    print("Loaded. Skipping fine-tune.")


In [ ]:
if FINETUNE_FROM_SCRATCH:
    os.makedirs(FINETUNED_WEIGHTS_DIR, exist_ok=True)
    base_model.save_weights(weights_path)
    print(f"Saved fine-tuned weights → {weights_path}")


In [ ]:
def plot_history(hist, title="", save_path=None):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m in zip([a1, a2], ["loss", "accuracy"]):
        train_key = m
        val_key   = f"val_{m}"
        if train_key not in hist.history:
            candidates = [k for k in hist.history if k.endswith(f"_{m}") and not k.startswith("val_")]
            if candidates:
                train_key = candidates[-1]
                val_key   = f"val_{train_key}"
        ax.plot(hist.history[train_key], label="train")
        ax.plot(hist.history[val_key],   label="val")
        ax.set(xlabel="Epoch", ylabel=m.title(), title=f"{title}{m.title()}")
        ax.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Plot saved to {save_path}")
    plt.show()

if history is not None:
    plot_history(history, f"{MODEL_CFG['name']} base — ", save_path=OUTPUT_DIR / f"{MODEL_CFG['name']}_base_training_history.png")
else:
    print("Skipped base training plot — loaded weights from Drive.")


In [ ]:
base_model.summary()

In [ ]:
backbone = None
for layer in base_model.layers:
    if isinstance(layer, tf.keras.Model):
        backbone = layer
        break

if backbone:
    backbone.summary()
else:
    print("Could not find the backbone (Functional model) within base_model.")

In [ ]:
def make_exit_head(features, name, exit_idx=1):

    if exit_idx == 2 or exit_idx == 3:
        x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu",
                                   name=f"conv1_{name}")(features)
        x = tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu",
                                   name=f"conv2_{name}")(x)
        x = tf.keras.layers.GlobalAveragePooling2D(name=f"gap_{name}")(x)
        x = tf.keras.layers.Dense(128, activation="relu", name=f"dense_mid_1_{name}")(x)
        x = tf.keras.layers.Dense(64, activation="relu", name=f"dense_mid_2_{name}")(x)

    elif exit_idx == 1:
        x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu",
                                   name=f"conv1_{name}")(features)
        x = tf.keras.layers.GlobalAveragePooling2D(name=f"gap_{name}")(x)
        x = tf.keras.layers.Dense(128, activation="relu", name=f"dense_mid_1_{name}")(x)
        x = tf.keras.layers.Dense(64, activation="relu", name=f"dense_mid_2_{name}")(x)

    x = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name=name)(x)
    return x


### Exit placement

Only the BB core has access to the dot-product accelerator. Backbone
MACs that are accelerator-eligible (pointwise Conv2D, Dense) are weighted
by `ALPHA_ACC < 1`; all other backbone MACs and **all** exit-head MACs
are at full software cost (the EE core executes the heads without
accelerator access). The constraint *EE-head-time < BB-segment-time*
must hold for every exit, including the last one — for the last exit,
the relevant segment is the backbone tail from the exit to the end.

If no feasible configuration exists at `N_EXITS_TARGET`, the algorithm
reduces N by one and retries, down to `N_EXITS_MIN`. Set
`ALPHA_ACC = 1.0` if accelerator is absent from the SoC.


In [ ]:
if AUTO_SELECT_EXITS:
    print("AUTO_SELECT_EXITS = True — running exit placement algorithm ...\n")

    N_EXITS_TARGET   = 3
    N_EXITS_MIN      = 1
    ALPHA_ACC        = 0.25   # accelerator effective-cost factor (1.0 = no accel)

    MIN_FIRST_DEPTH  = 10.0   # first exit must lie past this %
    LAST_DEPTH_RANGE = (55.0, 90.0)
    TARGET_LAST_DEPTH= 70.0
    LAMBDA_BAL       = 10.0   # weight of segment-balance term in score

    # MAC accounting
    # split_layer_macs(): returns (accel_eligible_macs, sw_only_macs).
    # Eligible operator type for AHBconv accelerator are pointwise (1x1) Conv2D and Dense.

    def split_layer_macs(layer):
        out = layer.output_shape
        inp = layer.input_shape
        if isinstance(inp, list): inp = inp[0]
        name = type(layer).__name__
        if name == "Conv2D":
            k = layer.kernel_size
            macs = k[0]*k[1]*inp[-1]*layer.filters*out[1]*out[2]
            if k == (1, 1):
                return macs, 0          # pointwise -> accelerator
            return 0, macs              # spatial kxk -> CPU
        if name == "DepthwiseConv2D":
            k = layer.kernel_size
            macs = k[0]*k[1]*inp[-1]*layer.depth_multiplier*out[1]*out[2]
            return 0, macs              # depthwise -> CPU
        if name == "Dense":
            return inp[-1]*layer.units, 0   # dense -> accelerator
        if name == "GlobalAveragePooling2D":
            return 0, inp[1]*inp[2]*inp[3]
        if name == "BatchNormalization":
            return 0, 4*out[-1]
        return 0, 0

    def backbone_effective_cost(layer, alpha=ALPHA_ACC):
        a, s = split_layer_macs(layer)
        return alpha*a + s

    def head_raw_cost(layer):
        a, s = split_layer_macs(layer)
        return a + s

    # Cumulative backbone effective cost
    layer_costs = [backbone_effective_cost(l) for l in backbone.layers]
    cumulative  = np.cumsum(layer_costs)
    total_bb    = cumulative[-1]
    print(f"Backbone: {len(backbone.layers)} layers, "
          f"{total_bb:,.0f} effective ops (alpha_acc={ALPHA_ACC})\n")

    # Candidate exit points
    BOUNDARY_KEYWORDS = ["Add", "project/BatchNorm", "_pool", "_concat",
                         "block_pool", "_out", "pool_conv"]
    candidates = []
    for i, layer in enumerate(backbone.layers):
        shape = layer.output_shape
        if isinstance(shape, list): shape = shape[0]
        if len(shape) != 4 or shape[1] is None:
            continue
        is_boundary = any(kw in layer.name for kw in BOUNDARY_KEYWORDS)
        candidates.append({
            "idx": i, "name": layer.name, "shape": shape,
            "cum_cost": cumulative[i],
            "depth_pct": cumulative[i] / total_bb * 100,
            "is_boundary": is_boundary,
        })

    boundary_cands = [c for c in candidates if c["is_boundary"]]
    if len(boundary_cands) >= 2 * N_EXITS_TARGET:
        candidates = boundary_cands

    print(f"Candidate exit points: {len(candidates)}")
    for c in candidates:
        tag = " *" if c["is_boundary"] else ""
        print(f"  [{c['idx']:3d}] {c['name']:<50s} "
              f"shape={c['shape'][1:]}  depth={c['depth_pct']:5.1f}%{tag}")
    print()

    # Exit-head MAC cost at each candidate
    ee_head_cost = {}
    for c in candidates:
        feat_shape = c["shape"][1:]
        dummy_input = tf.keras.Input(shape=feat_shape)
        for eidx in range(1, N_EXITS_TARGET + 1):
            dummy_out = make_exit_head(dummy_input,
                                       f"probe_{c['idx']}_e{eidx}",
                                       exit_idx=eidx)
            probe = tf.keras.Model(inputs=dummy_input, outputs=dummy_out)
            ee_head_cost[(c["idx"], eidx)] = sum(head_raw_cost(l)
                                                 for l in probe.layers)
            del probe

    from itertools import combinations

    def search_n_exit(N):

        """Return list of valid configs for exactly N exits, sorted by score."""

        if len(candidates) < N:
            return []
        results = []
        for combo in combinations(range(len(candidates)), N):
            cs = [candidates[i] for i in combo]
            depths = [c["depth_pct"] for c in cs]

            if depths[0] < MIN_FIRST_DEPTH:
                continue
            if not (LAST_DEPTH_RANGE[0] <= depths[-1] <= LAST_DEPTH_RANGE[1]):
                continue

            segs = [cs[k+1]["cum_cost"] - cs[k]["cum_cost"]
                    for k in range(N - 1)]
            segs.append(total_bb - cs[-1]["cum_cost"])
            if any(s <= 0 for s in segs):
                continue

            head_costs = [ee_head_cost[(cs[k]["idx"], k+1)] for k in range(N)]

            if any(head_costs[k] >= segs[k] for k in range(N)):
                continue

            r = min(segs) / max(segs)
            dist  = abs(depths[-1] - TARGET_LAST_DEPTH)
            score = dist - LAMBDA_BAL * r

            results.append({
                "n_exits": N,
                "exits": [c["name"] for c in cs],
                "depths": depths,
                "segments": segs,
                "head_costs": head_costs,
                "margins": [segs[k] / head_costs[k] for k in range(N)],
                "score": score,
            })
        results.sort(key=lambda x: x["score"])
        return results

    # Iterative reduction
    chosen = None
    for N in range(N_EXITS_TARGET, N_EXITS_MIN - 1, -1):
        print(f"{'='*80}\nTrying N = {N} exit(s) ...\n{'='*80}")
        found = search_n_exit(N)
        print(f"  Found {len(found)} valid configurations at N={N}")
        if found:
            chosen = found[0]
            for rank, combo in enumerate(found[:5], 1):
                print(f"\n  -- Option {rank} (N={N}) --")
                for ei in range(N):
                    print(f"    Exit {ei+1}: {combo['exits'][ei]:<50s} "
                          f"({combo['depths'][ei]:.1f}%)")
                print(f"    BB-seg (eff): "
                      + "   ".join(f"S{k+1}={s:,.0f}"
                                   for k, s in enumerate(combo['segments'])))
                print(f"    EE-head (raw):"
                      + "   ".join(f"H{k+1}={h:,.0f}"
                                   for k, h in enumerate(combo['head_costs'])))
                print(f"    Margins (S/H):"
                      + "   ".join(f"{m:.2f}x" for m in combo['margins']))
                print(f"    Score: {combo['score']:.3f}")
            break
        else:
            print(f"  No feasible placement with {N} exits — reducing N.\n")

    if chosen is not None:
        MODEL_CFG["exit_layers"] = chosen["exits"]
        MODEL_CFG["n_exits_used"] = chosen["n_exits"]
        print(f"\n{'='*80}")
        print(f"Selected N={chosen['n_exits']} exit(s): {chosen['exits']}")
        print(f"MODEL_CFG['exit_layers'] and ['n_exits_used'] updated.")
        print(f"{'='*80}")
    else:
        print(f"\nERROR: no feasible placement for any N in "
              f"[{N_EXITS_MIN}, {N_EXITS_TARGET}]. "
              f"Falling back to exit_layers from registry.")

else:
    print(f"AUTO_SELECT_EXITS = False — using exit_layers from registry:")
    print(f"  {MODEL_CFG['exit_layers']}")


## 4 · Attach early-exit heads

The backbone is frozen; only exit heads are trained jointly.


In [ ]:
def build_multi_exit_model(base_model, cfg):
    backbone = None
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.Model):
            backbone = layer
            break
    backbone.trainable = False

    inp = tf.keras.Input(shape=IMAGE_SIZE)
    if cfg.get("expects_0_255"):
        x = tf.keras.layers.Rescaling(255.0, name="rescale_to_255")(inp)
    else:
        x = inp

    backbone_out = backbone(x)

    exits = []
    for i, layer_name in enumerate(cfg["exit_layers"], start=1):
        feat = backbone.get_layer(layer_name).output
        exits.append(make_exit_head(feat, f"early_exit_{i}", exit_idx=i))

    x_final = backbone.output
    backbone_idx = base_model.layers.index(backbone)
    for layer in base_model.layers[backbone_idx + 1:]:
        x_final = layer(x_final)
    exits.append(x_final)

    return tf.keras.Model(inputs=backbone.input, outputs=exits,
                          name=f"{cfg['name']}_multi_exit")


multi_exit_model = build_multi_exit_model(base_model, MODEL_CFG)
multi_exit_model.summary()
print("Outputs:", multi_exit_model.output_names)


In [ ]:
output_names = multi_exit_model.output_names
losses  = {n: "categorical_crossentropy" for n in output_names}
weights = dict(zip(output_names, EE_LOSS_WEIGHTS))

model_save_path = pathlib.Path(FINETUNED_WEIGHTS_DIR).parent / OUTPUT_DIR.name / f"{MODEL_CFG['name']}_multi_exit.h5"
print(f"Attempting to load/save model at: {model_save_path}")

if model_save_path.exists():
    print(f"Loading multi-exit model from {model_save_path}...")
    multi_exit_model = tf.keras.models.load_model(str(model_save_path))
    ee_history = None
    print("Model loaded.")
else:
    print("Multi-exit model not found. Training...")
    model_save_path.parent.mkdir(parents=True, exist_ok=True)
    multi_exit_model.compile(
        optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
        loss=losses, loss_weights=weights, metrics=["accuracy"],
    )
    ee_history = multi_exit_model.fit(train_ds, validation_data=val_ds,
                                      epochs=10, verbose=2)
    multi_exit_model.save(str(model_save_path))
    print(f"Multi-exit model saved to {model_save_path}.")

In [ ]:
plot_history(ee_history, f"{MODEL_CFG['name']} multi-exit — ", save_path=OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_training_history.png")

## 5 · INT8 quantisation

In [ ]:
def representative_dataset():
    for batch, _ in full_ds.take(100):
        yield [batch]

# Unquantized TFLite conversion
unquantized_converter = tf.lite.TFLiteConverter.from_keras_model(multi_exit_model)
unquantized_tfl_path = OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_unquantized.tflite"
unquantized_model = unquantized_converter.convert()
unquantized_tfl_path.write_bytes(unquantized_model)
print(f"Unquantized TFLite → {unquantized_tfl_path} ({len(unquantized_model):,} bytes)")

quantized_tfl_path = None
if QUANTIZE_FOR_TVM:
    # Quantized TFLite conversion
    converter = tf.lite.TFLiteConverter.from_keras_model(multi_exit_model)
    converter.optimizations             = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset    = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type      = tf.float32
    converter.inference_output_type     = tf.float32

    quantized = converter.convert()
    quantized_tfl_path = OUTPUT_DIR / f"{MODEL_CFG['name']}_multi_exit_int8.tflite"
    quantized_tfl_path.write_bytes(quantized)
    print(f"Quantized TFLite → {quantized_tfl_path} ({len(quantized):,} bytes)")
else:
    print("Skipping INT8 quantization as QUANTIZE_FOR_TVM is False.")

tflite_to_compile_path = quantized_tfl_path if QUANTIZE_FOR_TVM else unquantized_tfl_path
print(f"Model selected for TVM compilation: {tflite_to_compile_path}")


## 6 · Compile to C source

In [ ]:
import time

model_file_path = tflite_to_compile_path

print(f"Loading TFLite model from: {model_file_path}")
with open(model_file_path, 'rb') as f:
    tfl_model_bytes = f.read()

try:
    import tflite
    tfl_model = tflite.Model.GetRootAsModel(tfl_model_bytes, 0)
except AttributeError:
    import tflite.Model
    tfl_model = tflite.Model.Model.GetRootAsModel(tfl_model_bytes, 0)

mod, params = tvm.relay.frontend.from_tflite(tfl_model)
target   = tvm.target.Target("c")
runtime  = Runtime("crt")
executor = Executor("aot", {"unpacked-api": True, "interface-api": "c",
                            "workspace-byte-alignment": 8})

if params:
    mod["main"] = tvm.relay.build_module.bind_params_by_name(mod["main"], params)

# ── Per-model compilation strategy ───────────────────────────
# Most models use the custom 3-phase pipeline (fold-before-QNN)
# at opt_level 3. Architectures like NASNet need the
# standard pipeline at a lower opt_level — see model registry.
USE_CUSTOM   = MODEL_CFG.get("tvm_custom_pipeline", True)
OPT_LEVEL    = MODEL_CFG.get("tvm_opt_level", TVM_OPT_LEVEL)

t0 = time.time()

if USE_CUSTOM:
    print(f"Compiling with custom pipeline, opt_level={OPT_LEVEL} ...")
    disabled_qnn_passes = ["qnn.Legalize", "qnn.Canonicalize"]
    if not QUANTIZE_FOR_TVM:
        print("  Disabling QNN-specific passes for unquantized model.")

    with tvm.transform.PassContext(
            opt_level=OPT_LEVEL,
            disabled_pass=disabled_qnn_passes if not QUANTIZE_FOR_TVM else [],
            config={"tir.disable_vectorize": True}):
        pre_passes = tvm.transform.Sequential([
            relay.transform.InferType(),
            relay.transform.FoldConstant(),
            relay.transform.SimplifyInference(),
        ])
        mod = pre_passes(mod)

        if QUANTIZE_FOR_TVM:
            qnn_passes = tvm.transform.Sequential([
                relay.qnn.transform.Legalize(),
                relay.qnn.transform.CanonicalizeOps(),
            ])
            mod = qnn_passes(mod)

        compiled = tvm.relay.build(mod, target, runtime=runtime,
                                   executor=executor, params={})
else:
    print(f"Compiling with standard pipeline, opt_level={OPT_LEVEL} ...")
    disabled_passes = ["relay.AlterOpLayout"]
    if not QUANTIZE_FOR_TVM:
        print("  Disabling QNN-specific passes for unquantized model.")
        disabled_passes.extend(["qnn.Legalize", "qnn.Canonicalize"])

    with tvm.transform.PassContext(
            opt_level=OPT_LEVEL,
            disabled_pass=disabled_passes,
            config={"tir.disable_vectorize": True}):
        compiled = tvm.relay.build(mod, target, runtime=runtime,
                                   executor=executor, params={})

elapsed = time.time() - t0
print(f"Compilation finished in {elapsed:.1f}s")
param_size = len(tvm.runtime.save_param_dict(params)) if params else len(tvm.runtime.save_param_dict(compiled.get_params()))
print(f"Params: {param_size:,} bytes")
tar_suffix = "_int8_aot.tar" if QUANTIZE_FOR_TVM else "_float32_aot.tar"
tar = OUTPUT_DIR / f"{MODEL_CFG['name']}_EE{tar_suffix}"
export_model_library_format(compiled, tar)
print(f"MLF archive → {tar}")
